# TSDAE.py

This notebook performs unsupervised TSDAE domain adaptation on a pre-trained sentence transformers model. TSDAE, or Transformer-based Sentence De-noising Auto-Encoding, is a means of adapting an existing language model to a new, technical domain by teaching it updated context vectors for words and sentences. Examples of data are supplied with key words missing (referred to as "noisy" data), and the model is tasked with reconstructing the missing input, which it learns to associate with the difference in vector representations of the noisy and reconstructed sentences. Using a collection of sentences that were assembled from cross-walked, extracted text from LOINC Part Descriptions and the RELMA database, the notebook constructs an in-memory dataset of pairs of sentences and noisy transformations of those sentences. The notebook instantiates a `sentence-transformers` model, configures the properties of its hidden layer for optimized training, and then applies batched TSDAE.

The output of this script will save one or more directories into the notebook's local workspace. One of them will be a `trainer_output` directory, which contains information used as part of logging and metricization while actually batching TSDAE. It can be safely deleted after the script completes. The other directory will consist of the new data and weight layers for the updated, freshly trained model. This directory should be kept in working memory as long as the model is being embedded and evaluated. This is because when invoking `model = SentenceTransformer()`, `sentence-transformers` first checks local memory for a model directory with that name. Finding one in the notebook's working memory will allow it to load directly without reaching out to HuggingFace (which would actually cause an error, as we've updated the model's name signature heavily).


## Setup

Make sure that once the compute instance is running, you activate the kernel associated with the DIBBs Env in the upper right dropdown. Its packages are correctly optimized for this notebook and avoids some `numpy` instabilities plaguing Azure.

Additionally, the `Datasets` package used to prepare and batch the TSDAE examples requires a particular optimization back-end; we need to pin specific versions here so that we can get the right compatibility.

In [ ]:
pip install azure-keyvault-secrets azure-identity azure-ai-ml azureml-fsspec nltk datasets 'transformers>=4.51.1' 'accelerate>=0.26.0'

TSDAE is designed to key off the word- and punctuation-based tokenization of NLTK, so we need to make sure we instantiate those objects and download any relevant data sets (notably the punctuation tagger).

In [ ]:
import nltk
nltk.download('punkt_tab')

Now we'll do our basic, standard authentication work. We need all these variables to be able to access our container storage from a file mount. The `DATASTORE_NAME` is not a protected secret and therefore doesn't need to be stashed in KeyVault, as it's a standard Azure default.

In [ ]:
# Authenticate to Key Vault
from azure.identity import DefaultAzureCredential
from azure.keyvault.secrets import SecretClient

credential = DefaultAzureCredential()
key_vault = "dibbsttc6059789213"
secret_client = SecretClient(vault_url=f"https://{key_vault}.vault.azure.net/", credential=credential)

SUBSCRIPTION = secret_client.get_secret("subscription").value
RESOURCE_GROUP = secret_client.get_secret("resource-group").value
WS_NAME = secret_client.get_secret("workspace-name").value
DATASTORE_NAME = 'workspaceblobstore'

_IMPORTANT_: This step can't be skipped, even though we're not directly using any of the `ml_client` functionality. This authentication and connection step allows us to use this notebook cleanly within our compute ecosystem. Basically, instantiating the class object acts as a connection that allows us to do everything that follows.

In [ ]:
from azure.ai.ml import MLClient

# Authenticate and connect to workspace
ml_client = MLClient(
    DefaultAzureCredential(),
    SUBSCRIPTION,
    RESOURCE_GROUP,
    WS_NAME,
)

Finally, TSDAE is only feasible with a compute instance attached to GPU. This cell ensures that the GPU is available for CUDA optimization.

In [ ]:
import torch
assert torch.cuda.is_available()

## Step 1: Create File Mount

The Azure Machine Learning File Mount system, though cumbersomely named, allows us to _directly_ access files and objects we have stored in the DIBBs TTC container. Any `.txt` or `.csv` files need to be created as Data Assets (see sidebar on left), so make sure the SNOINC extracts file is properly instantiated as an Azure resource before loading it.

In [ ]:
from azureml.fsspec import AzureMachineLearningFileSystem

# Instantiate a file system over the workspace so we can interact with data
# assets directly--we get all the goodies like open, ls, etc.
fs = AzureMachineLearningFileSystem(
    f"azureml://subscriptions/{SUBSCRIPTION}/resourcegroups/{RESOURCE_GROUP}/workspaces/{WS_NAME}/datastores/{DATASTORE_NAME}"
)

## Step 2: Load TSDAE Sentences

Using our file mount, we can read the list of sentences we have curated from the LOINC part descriptions from Blob Storage. Remember, the LOINC file needs to already be an Azure Data Asset before this will work. We decode the bytestrings into proper UTF-8, then append all sentences that have more than 10 tokens into a list for model processing. Enforcing a minimum tokens limit ensures there is enough surrounding context in the masked, noisy version for the model to make an informed reconstruction.

Our work has found that _fewer_ TSDAE sentences leads to less overfitting during training, which means better performance during evaluation. By default, we recommend using the `tsdae_data_10k.txt` file for training, but if desired, we do have a `tsdae_data_full.txt` with 27k to 30k examples (depending on desired `MIN_TOKENS_PER_EXAMPLE`).

In [ ]:
TSDAE_FILE = "tsdae_data_10k.txt"
MIN_TOKENS_PER_EXAMPLE = 10

sentences = []

print("Loading TSDAE examples...")
with fs.open(TSDAE_FILE) as fp:
    for line in fp:
        # Blob storage is bytes-based, so we need to decode before string operations
        line_str = line.decode("utf-8")
        if line_str.strip() != "" and len(line_str.strip().split()) >= MIN_TOKENS_PER_EXAMPLE:
            sentences.append(line_str.strip())

# There are either ~10k examples or ~30k examples in this list, depending
# on whether the abridged set or full validation set is used
assert len(sentences) >= 10000
print(f"{len(sentences)} LOINC sentences loaded.")

## Step 3: Create Batched Dataset

TSDAE involves batching a validation dataset into small minibatches and then feeding them to the model for gradient-update training. The process is exactly the same as in regular training: 8/16/32 randomly selected examples are given to the model, the model makes predictions for their vector versions, the error loss between the model's prediction and the right answer is calculated, and this difference is scaled down and used to update the internal model weights.

The difference with TSDAE is that the "right" answer isn't known ahead of time. To calculate the loss and update its weights, we apply a "noisification" to the data, blanking out some of the words to force the model to make predictions based on the remaining context of the sentences. This prediction can then be comparaed against the full, non-noisy version of the sentence, and the difference between the two vector projections is used to update the model's internal weights. This "blanking out" is what allows TSDAE to learn new weights for technical, domain specific terms: when clinical words are blanked out of the noisy input, the model's noisy prediction is forced to use different context then if the clinical word were regularly processed, leading to more intelligent weights for words that aren't otherwise used in regular English.

Here, we're creating a lambda function that will be applied during batch feeding of the training data to the model. The most important parameter is the `DEL_RATIO` value, which assigns the probability that a given word in the noisy input will be randomly selected for blanking. Higher values of this ratio lead to noisier training data, which can improve context updating but can also lead to "catastrophic forgetting"--if the data is too noisy, then the remaining context won't have enough meaning left for the model to make informed predictions, which will actually lead to vector degradation as updates are worse than original values. Over time, this pushes all vectors to look the same, which leads to terrible performance. The TSDAE authors found that a value of 0.6 worked best for them, but we've experimentally found that values of 0.3 or 0.45 work better for our case, due to the length and technicality of the LOINC domain sentences.


In [ ]:
import random

from datasets import Dataset
from nltk import word_tokenize
from nltk.tokenize.treebank import TreebankWordDetokenizer

# Controls the probability that a given word in the input sentence will be
# deleted
DEL_RATIO = 0.3

# This function will be lambda-tized as part of a set compression when creating
# the data dictionary. It will be applied to each text entry in the list of 
# sentences and will produce a "noisy" variant of the text, subject to 
# minimum usability criteria.
def noise_transform(batch, del_ratio=0.6):
    noisy_texts = []
    for text in batch["text"]:
        words = word_tokenize(text)
        if len(words) == 0:
            # Special case: if the tokenizer can't parse the split, it's likely
            # that punctuation in the input is causing confusion over something
            # medically-formatted (a chemical compound or molecular chain, for 
            # example). In that case, leave the text be.
            noisy_texts.append(text)
            continue

        kept_words = [word for word in words if random.random() < del_ratio]
        # Guarantee that at least one word remains
        if len(kept_words) == 0:
            noisy_texts.append(random.choice(words))
            continue
        noisy_texts.append(TreebankWordDetokenizer().detokenize(kept_words))

    return {"noisy": noisy_texts, "text": batch["text"]}

# Scale the size of the train-test split based on how many sentences 
# we have originally
test_size = int(float(len(sentences)) * 0.2)
print(f"Using test set size of {test_size}")

# We use can use `set_transform` here so that the noisy text differs
# each time a text example is batched (which improves robustness)
dataset = Dataset.from_dict({"text": sentences})
dataset.set_transform(
    transform=lambda batch: noise_transform(batch, DEL_RATIO), columns=["text"], output_all_columns=True
)

# We'll also split out a testing size that lets us validate as we go
dataset = dataset.train_test_split(test_size=test_size)
train_dataset = dataset["train"]
eval_dataset = dataset["test"]

## Step 4: Construct Model

With the data prepared, we need to instantiate the model and prepare our loss function. The only significant point here is the parameter `tie_encoder_decoder=True`. Normally, these LLMs are "bidirectional"--they process input sequences both forwards and backwards to absorb context from both directions, and they both encode (transform input into vectors) and decode (transform vectors back into string estimates) that context for loss calculations. For our purposes, we don't want to change the decoding procedure of how vectors are mapped back into strings. We only want to change the encoding of how strings are vectorized in the first place. This parameter allows us to fully train just some of the weights in the model.

In [ ]:
from sentence_transformers import models, SentenceTransformer
from sentence_transformers.losses import DenoisingAutoEncoderLoss

# Model-based training architecture parameters
MODEL_NAME = "intfloat/e5-large-v2"

model = SentenceTransformer(MODEL_NAME)
tokenizer = model.tokenizer

# Note that this will likely result in warnings as we're loading 'model_name' as a decoder, but it likely won't
# have weights for that yet. This is fine, as we'll be training it from scratch.
train_loss = DenoisingAutoEncoderLoss(model, decoder_name_or_path=MODEL_NAME, tie_encoder_decoder=True)


Once the base model is instantiated from HuggingFace, we'll need to manually initialize the internal attention layers. These layers are responsible for mapping each word in the input into the appropriate token ID in the model's internal representation of the data. For example, the string "Fact: bears eat beets" might map to the sequence `[184 24673 99745 37721]`. An important task for these layers is to "pad" all input sequences to be the same length for vector processing, meaning all sentences less than the internal `MAX_SEQ_LENGTH` have `<PAD>` tokens appended until they reach maximum length. This padding task is lazy by default, meaning it won't initialize until data has actually passed through the model. This generates some annoying warnings and might lead to the first batch of input being "corrupted" by non-padding, so we'll just get that first batch out of the way here.

In [ ]:
x = train_dataset[150]['text']
assert x is not None and x != ""
y = tokenizer(x)
z = tokenizer.decode(y['input_ids'])
assert z is not None and z != ""

print("Tokenizer masks initialized from lazy matrices")

## Step 5: Train the Model

The last thing we need to do before executing training is prepare our hyperparameter arguments. There are a lot of values that can be manipulated here, but the key ones are called out as all-caps constants in the next cell.

* `LEARNING_RATE` dictates how much the model scales down gradient updates during loss calculation. Lower values of this (e.g. 1e-6 or 1e-5) lead to smaller, smoother updates over time, which are more stable than larger values (1e-4 or 1e-3) but which might not capture all the details. Generally, it is easy enough to overfit with TSDAE that we strongly suggest small values.
* `NUM_EPOCHS` dictates how many epochs (full passes through the training data) to perform. It is almost always recommended to perform only a single TSDAE epoch, as otherwise it is too easy for the model to forget its original, general-purpose training.
* `TRAIN_BATCH_SIZE` indicates how many example sentences should make up one of the mini-batches fed to the model. There is no right value for this size, as the most significant impact of this variable is processing efficiency and capability. Small LLMs (under 1B params) can use a size of 32 to make training fast and smooth. Medium and Large LLMs may need batch sizes of 8 or even 4 in order to fit all the parameter overhead in memory.
* `NUM_EVAL_BENCHMARKS` indicates how many validation steps to perform during training. Each validation step will produce a calculation of the loss function on the set-aside testing portion of the input data. It will create a new row in the output table that shows the plotted loss over time. More validation steps can help visually demonstrate the best points in training, but validation calculation is slow and each additional validation benchmark can dramatically lengthen training time.


In [ ]:
from sentence_transformers.training_args import SentenceTransformerTrainingArguments

LEARNING_RATE = 1e-5
NUM_EPOCHS = 1
TRAIN_BATCH_SIZE = 32
NUM_EVAL_BENCHMARKS = 5
BATCHES_PER_EVAL = int(float(len(sentences) - test_size) / TRAIN_BATCH_SIZE / 5)
UPDATED_NAME = f"{MODEL_NAME.replace('/', '_')}_{DEL_RATIO}_{str(LEARNING_RATE).replace('-', '')}"

print(f"Using {BATCHES_PER_EVAL} batches per validation step.")


# 5. Define the training arguments
args = SentenceTransformerTrainingArguments(
    # Required parameter:
    output_dir=None,
    # Optional training parameters:
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=TRAIN_BATCH_SIZE,
    warmup_ratio=0.15,
    fp16=False,  # Set to False if you get an error that your GPU can't run on FP16
    bf16=True,  # Set to True if you have a GPU that supports BF16
    # Optional tracking/debugging parameters:
    eval_strategy="steps",
    eval_steps=BATCHES_PER_EVAL,
    save_strategy="steps",
    save_steps=BATCHES_PER_EVAL,
    save_total_limit=NUM_EVAL_BENCHMARKS,
    logging_steps=100,
    run_name="tsdae",  # Will be used in W&B if `wandb` is installed
)

Finally, we'll train the model and save the output.

In [ ]:
from sentence_transformers.trainer import SentenceTransformerTrainer

# 6. Create the trainer & start training
trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    loss=train_loss,
)
trainer.train()
model.save(f"{UPDATED_NAME}/")

Optionally, we can also save this model to remote storage if we intend to analyze it or continue training in the future.

In [ ]:
import os
fs.put(UPDATED_NAME, f"/models/tsdae/{UPDATED_NAME}")